In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.bolshanin.lesson2 import Exercise


def normalize_data(X_train, X_val, X_test):

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return X_train_norm, X_val_norm, X_test_norm


def evaluate_model(lr, batch_size, X_train, X_val, y_train, y_val, n_iter, seed):

    rng = np.random.default_rng(seed)
    model = Exercise.create_logistic_model(num_features=X_train.shape[1], rng=rng)

    Exercise.fit(model, X_train, y_train, lr=lr, n_iter=n_iter, batch_size=batch_size)

    auroc = model.metric(X_val, y_val, metric_type="auroc")

    return auroc, model


def main():

    print("=" * 60)
    print("Подбор гиперпараметров для логистической регрессии на Iris dataset")
    print("=" * 60)

    iris = load_iris()
    X = iris.data
    y = iris.target

    y_binary = (y < 1).astype(int)

    print("\n[1] Данные:")
    print(f"  Всего образцов: {len(X)}")
    print(f"  Количество признаков: {X.shape[1]}")
    print(f"  Класс 0 (Iris-setosa): {np.sum(y_binary == 1)} шт")
    print(f"  Класс 1 (Iris-versicolor + Iris-virginica): {np.sum(y_binary == 0)} шт")

    print("\n[2] Разделение данных:")

    X_train, X_temp, y_train, y_temp = train_test_split(X, y_binary, test_size=0.4, random_state=42, stratify=y_binary)

    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    print(f"  Train: {len(X_train)} образцов")
    print(f"  Val:   {len(X_val)} образцов")
    print(f"  Test:  {len(X_test)} образцов")

    print("\n[3] Нормализация данных:")
    X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

    print("  Средние значения после нормализации (train):")
    print(f"    {X_train_norm.mean(axis=0)}")
    print("  Стандартные отклонения после нормализации (train):")
    print(f"    {X_train_norm.std(axis=0)}")

    print("\n[4] Подбор гиперпараметров:")

    n_iter = 25

    lr_values = [0.001, 0.002, 0.003, 0.004, 0.005, 0.007, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1]
    batch_size_values = [1, 2, 4, 8, 16, 32, 64, None]

    seeds = [2, 4, 8, 16, 32]

    best_config = {
        "lr": 0,
        "batch_size": None,
        "auroc_mean": 0.0,
        "auroc_std": 1.0,
    }

    print(
        f"  Поиск по {len(lr_values)} × {len(batch_size_values)} = "
        f"{len(lr_values) * len(batch_size_values)} комбинациям"
    )
    print(f"  Каждая комбинация тестируется на {len(seeds)} random seed")
    print(f"  Всего экспериментов: {len(lr_values) * len(batch_size_values) * len(seeds)}")
    print("\n  " + "-" * 50)

    for lr in lr_values:
        for batch_size in batch_size_values:
            auroc_scores = []

            for seed in seeds:
                auroc, _ = evaluate_model(lr, batch_size, X_train_norm, X_val_norm, y_train, y_val, n_iter, seed)
                auroc_scores.append(auroc)

            auroc_mean = np.mean(auroc_scores)
            auroc_std = np.std(auroc_scores)

            print(f"  LR={lr:6.4f}, BS={str(batch_size):4s} -> AUROC = {auroc_mean:.4f} ± {auroc_std:.4f}")

            if (auroc_mean > best_config["auroc_mean"]) or (
                auroc_mean == best_config["auroc_mean"] and auroc_std < best_config["auroc_std"]
            ):
                best_config.update(
                    {
                        "lr": lr,
                        "batch_size": batch_size,
                        "auroc_mean": auroc_mean,
                        "auroc_std": auroc_std,
                    }
                )

    print("\nВсего образцов: 150")
    print(f"Класс 0: {np.sum(y_binary == 1)} шт, Класс 1 + 2: {np.sum(y_binary == 0)} шт")
    print("Best parameters:")
    print(f"Learning Rate: {best_config['lr']}")
    print(f"Batch Size: {best_config['batch_size']}")
    print(f"Val AUROC: {best_config['auroc_mean']:.4f} +- {best_config['auroc_std']:.4f}")
    print(f"Epochs: {n_iter}")

    return best_config


if __name__ == "__main__":
    best_config = main()